# Ingénierie des caractéristiques I

Ce notebook implémente les étapes demandées pour la **Partie 2** du Jalon 2

## 1. Organisation et partitionnement des données

Nous utilisons le fichier `data/processed/shots.csv` généré à l'Étape 1 qui contient déjà les tirs et buts extraits pour toutes les saisons 2016-2017 à 2023-2024.

Comme il est demandé, nous allons d'abord séparé les données en 3 ensembles fixes. 
- Test : On réserve la saison 2023/2024 pour les tests finaux
- Entraînement : On prend 80% dans les saisons restantes 
- Validation : On prend 20% dans les saisons restantes

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from ift6758.data_acquisition import PROJECT_ROOT
from ift6758.features.build_dataset import PROCESSED


data_path = PROJECT_ROOT / "data" / "processed" / "shots.csv"
df_all = pd.read_csv(data_path)

df_test_regular = df_all[(df_all["game_type"] == 2) & (df_all["season"] == 20232024)]
df_test_playoffs = df_all[(df_all["game_type"] == 3) & (df_all["season"] == 20232024)]

df_regular_dev = df_all[(df_all["game_type"] == 2) & (df_all["season"] != 20232024)]

# Ici on sépare les saisons réservées au développement en deux ensemble, on 
# doit s'assurer que les ensembles soient stratifiés sur is_goal pour ne pas 
# avoir de déséquilibre.
train_df, val_df = train_test_split(
    df_regular_dev,
    test_size=0.20,
    stratify=df_regular_dev['is_goal'],
    random_state=42
)

print("Proportion buts (Train) :", train_df['is_goal'].mean())
print("Proportion buts (Val)   :", val_df['is_goal'].mean())

PROCESSED.mkdir(parents=True, exist_ok=True)

csv_file = PROCESSED / 'test_regular_20232024.csv'
df_test_regular.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'test_playoffs_20232024.csv'
df_test_playoffs.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'train_set.csv'
train_df.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'validation_set.csv'
val_df.to_csv(csv_file, index=False)

Proportion buts (Train) : 0.09736637600179261
Proportion buts (Val)   : 0.09736352800953516


Dans notre jeux de données train_df nous avons toutes les caractéristiques que build_dataset inclus. Autrement dit nous respectons avoir les caractéristiques shot_distance, shot_angle, is_goal, empty_net.
Pour le reste de la section, nous allons définir un dataframe qui contient simplement ces colonnes. On va aussi afficher les valeurs manquantes de chaque colonnes :

In [6]:
df_cols_imp_train = train_df[["shot_distance", "shot_angle", "is_goal", "empty_net"]].copy()

# On va transformer empty_net pour que ce soit 0 ou 1 au lieu de True or False
df_cols_imp_train["empty_net"] = df_cols_imp_train["empty_net"].map({True: 1, False: 0})

print("Nombre de valeurs manquantes par caractéristique :")
print(df_cols_imp_train.isna().sum())


print(df_cols_imp_train.head(10))

Nombre de valeurs manquantes par caractéristique :
shot_distance      97
shot_angle       2354
is_goal             0
empty_net        3242
dtype: int64
        shot_distance  shot_angle  is_goal  empty_net
519593      54.120237   27.512003        0        0.0
544388      33.970576   42.614056        0        0.0
114441      38.470768   27.897271        0        0.0
250681      52.886671   15.350136        0        0.0
299334      46.097722    3.731397        0        0.0
359252      10.440307   16.699244        0        0.0
494789      25.961510   15.642246        0        0.0
449900      58.600341   25.253163        0        0.0
240518      15.652476   63.434949        0        0.0
490113      15.524175   14.931417        0        0.0
